# Step 1 — Data Loading and Sanity Check

IEEE SPC 2015 wrist PPG dataset.

**Three groups** (refined after inspecting the data):

| Group | Label | Activity | Source files | Count |
|-------|-------|----------|--------------|-------|
| T1 | Treadmill | walking/running, varying speeds | `DATA_01..12_TYPE01/02.mat` | 12 |
| T2 | Mixed arm exercise | shake, stretch, push-up, run, jump | `TEST_S*_T01.mat` | 4 |
| T3 | Boxing | intensive forearm/upper-arm boxing | `TEST_S*_T02.mat` | 6 |

**Row layout differs between training and test files** (handled transparently in cell 1.2):

| File type | `sig` shape | Row 0 | Row 1 | Row 2 | Row 3 | Row 4 | Row 5 |
|-----------|-------------|-------|-------|-------|-------|-------|-------|
| Training (`DATA_*`) | **6×N** | ECG | PPG ch1 | PPG ch2 | ACC x | ACC y | ACC z |
| Test (`TEST_*`)     | **5×N** | PPG ch1 | PPG ch2 | ACC x | ACC y | ACC z | — |

Ground truth is the variable `BPM0` in `_BPMtrace.mat` (training) / `True_*.mat` (test) — one value per 8 s window, 2 s shift.

**Consistent colors across all plots:** `T1 = steelblue`, `T2 = goldenrod`, `T3 = crimson`.

> **Kernel:** select `Python (ppg-entropy)` in the top-right kernel picker before running.
>
> Signal pre-processing (gravity removal, filtering, etc.) is deferred to a later step — this notebook works directly on the raw ACC signal so we can inspect its natural character.


In [ ]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import re

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Shared group → color map used by every plot in the project
GROUP_COLORS = {
    'T1': 'steelblue',   # treadmill
    'T2': 'goldenrod',   # mixed arm exercise
    'T3': 'crimson',     # boxing
}
GROUP_LABELS = {
    'T1': 'T1 — treadmill',
    'T2': 'T2 — mixed arm exercise',
    'T3': 'T3 — boxing',
}


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE = Path('hpi-dhc TROIKA main datasets-IEEE_SPC_2015')
TRAIN_DIR  = BASE / 'Training_data'
TEST_DIR   = BASE / 'TestData'
TRUEBPM_DIR = BASE / 'TrueBPM'

FS          = 125          # Hz
WIN_SAMPLES = 1000         # 8 s × 125 Hz
SHIFT       = 250          # 2 s × 125 Hz

## 1.1 — Build recording manifest

**Expected output:**
```
Total recordings: 22
  T1 — treadmill: 12 recordings
  T2 — mixed arm exercise: 4 recordings
  T3 — boxing: 6 recordings
```
followed by a list of all 22 recordings with IDs `01`–`22` and their filenames.

- IDs `01–12` → T1 (`DATA_*_TYPE*.mat`)
- IDs `13, 14, 19, 22` → T2 (`TEST_S*_T01.mat`)
- IDs `15, 16, 17, 18, 20, 21` → T3 (`TEST_S*_T02.mat`)
- No `AssertionError` (every `sig` file has a matching BPM counterpart)


In [ ]:
def build_manifest():
    """Return list of dicts: {rec_id, group, sig_path, bpm_path}.

    Groups:
      T1 — treadmill (training files DATA_*)
      T2 — mixed arm exercise (test files TEST_S*_T01 — shake/stretch/push-up/run/jump)
      T3 — boxing (test files TEST_S*_T02)
    """
    records = []

    # ── T1: Training files ─────────────────────────────────────────────────
    for sig_path in sorted(TRAIN_DIR.glob('DATA_*.mat')):
        if 'BPMtrace' in sig_path.name:
            continue
        stem     = sig_path.stem                       # e.g. DATA_01_TYPE01
        bpm_path = TRAIN_DIR / f'{stem}_BPMtrace.mat'
        assert bpm_path.exists(), f'Missing BPM file: {bpm_path}'
        rec_id = int(re.search(r'DATA_(\d+)', stem).group(1))
        records.append(dict(rec_id=rec_id, group='T1',
                            sig_path=sig_path, bpm_path=bpm_path))

    # ── T2/T3: Test files (T2 = T01 suffix, T3 = T02 suffix) ───────────────
    next_id = 13
    for sig_path in sorted(TEST_DIR.glob('TEST_S*.mat')):
        m = re.search(r'TEST_(S\d+)_(T\d+)', sig_path.name)
        subject_tag, task_tag = m.group(1), m.group(2)  # e.g. "S01", "T01"
        bpm_path = TRUEBPM_DIR / f'True_{subject_tag}_{task_tag}.mat'
        assert bpm_path.exists(), f'Missing BPM file: {bpm_path}'
        group = 'T2' if task_tag == 'T01' else 'T3'     # T01=mixed, T02=boxing
        records.append(dict(rec_id=next_id, group=group,
                            sig_path=sig_path, bpm_path=bpm_path))
        next_id += 1

    return records


manifest = build_manifest()
print(f'Total recordings: {len(manifest)}')
for grp in ('T1', 'T2', 'T3'):
    members = [r for r in manifest if r['group'] == grp]
    print(f'  {GROUP_LABELS[grp]}: {len(members)} recordings')
print()
for r in manifest:
    print(f"  [{r['rec_id']:02d}] {r['group']}  {r['sig_path'].name}")


## 1.2 — Load signals and compute per-window RMS ACC magnitude

`load_recording` returns a normalized dict `{ecg, ppg, acc, bpm0}` that hides the row-layout difference between training and test files.

The row indexing is driven by `sig.shape[0]` (not the group label) — more robust if a file ever gets misrouted. We also assert that T1 files are 6-row and T2/T3 files are 5-row to catch any future schema drift.

| File type | `sig` shape | ECG | PPG | ACC |
|-----------|-------------|-----|-----|-----|
| Training (T1) | 6×N | row 0 | rows 1–2 | rows 3–5 |
| Test (T2, T3) | 5×N | — | rows 0–1 | rows 2–4 |

**Expected output (22 lines, one per recording):**
- Format: `[id] group  duration  n_windows  n_gt  mean_acc_rms  (ECG=yes/no)`
- All 12 T1 rows show `ECG=yes`; all 10 T2/T3 rows show `ECG=no`
- Durations roughly 200–340 s
- `n_windows ≈ n_gt` (differ by ≤ 1)
- **Sanity check — updated expectation:** the three group means will overlap substantially (T1 ≈ 1.48, T2 ≈ 1.07, T3 ≈ 1.79). T3 boxing tends to match or exceed T1 treadmill, which is exactly the discrimination problem entropy is meant to solve.


In [ ]:
def load_recording(rec):
    """Load one recording and normalize the row layout.

    Training `sig` is 6×N: row 0 = ECG, rows 1-2 = PPG, rows 3-5 = ACC.
    Test `sig`    is 5×N: rows 0-1 = PPG, rows 2-4 = ACC (ECG withheld by competition).

    Row indexing is driven by sig.shape[0], so this works regardless of how the
    file is labelled. We then cross-check the shape against the expected group
    to catch any manifest/layout mismatch.
    """
    sig  = sio.loadmat(rec['sig_path'])['sig'].astype(float)
    bpm0 = sio.loadmat(rec['bpm_path'])['BPM0'].ravel().astype(float)
    n_rows = sig.shape[0]

    # Schema guard: T1 should always be 6-row, T2/T3 should always be 5-row.
    if rec['group'] == 'T1':
        assert n_rows == 6, (
            f"Expected 6 rows for T1 training file {rec['sig_path'].name}, got {n_rows}")
    else:
        assert n_rows == 5, (
            f"Expected 5 rows for {rec['group']} test file {rec['sig_path'].name}, got {n_rows}")

    if n_rows == 6:
        ecg = sig[0, :]
        ppg = sig[1:3, :]
        acc = sig[3:6, :]
    else:  # n_rows == 5
        ecg = None
        ppg = sig[0:2, :]
        acc = sig[2:5, :]
    return dict(ecg=ecg, ppg=ppg, acc=acc, bpm0=bpm0)


def rms_acc_windows(acc):
    """acc: 3×N array. Returns 1-D array of per-window RMS magnitude."""
    mag = np.sqrt(np.sum(acc**2, axis=0))   # instantaneous 3-axis magnitude
    n_windows = (acc.shape[1] - WIN_SAMPLES) // SHIFT + 1
    rms = np.array([
        np.sqrt(np.mean(mag[i*SHIFT : i*SHIFT + WIN_SAMPLES]**2))
        for i in range(n_windows)
    ])
    return rms


# Load all 22 recordings
recordings = []
for rec in manifest:
    data = load_recording(rec)
    rms  = rms_acc_windows(data['acc'])
    n_gt = len(data['bpm0'])
    recordings.append(dict(
        **rec,
        ecg=data['ecg'],
        ppg=data['ppg'],
        acc=data['acc'],
        bpm0=data['bpm0'],
        acc_rms_windows=rms,
        mean_acc_rms=rms.mean(),
        n_windows=len(rms),
        n_gt=n_gt,
    ))
    dur_s   = data['acc'].shape[1] / FS
    has_ecg = 'yes' if data['ecg'] is not None else 'no '
    print(f"[{rec['rec_id']:02d}] {rec['group']}  "
          f"{dur_s:6.1f}s  "
          f"{len(rms):4d} windows  "
          f"{n_gt:4d} GT  "
          f"mean ACC RMS={rms.mean():.3f}  "
          f"(ECG={has_ecg})")


## 1.3 — Mean ACC magnitude per recording (T1 / T2 / T3)

**Expected output:**
- Bar chart with 22 bars, x-axis labelled `1` … `22`, coloured by group (steelblue / goldenrod / crimson)
- Units are in **g** (≈ 1.0 is the resting-gravity baseline since raw ACC still contains the gravity DC component)
- Three group-mean lines printed below the plot
- **Expected pattern:** the three group means will be *similar* (approximately T1 ≈ 1.48, T2 ≈ 1.07, T3 ≈ 1.79). T3 boxing tends to match or exceed T1 treadmill, which is exactly the discrimination problem entropy is meant to solve.
- If T1 > T3 by a wide margin, or any group is near 0, flag it — that would indicate a loading or grouping bug.


In [ ]:
rec_ids  = [r['rec_id']       for r in recordings]
mean_rms = [r['mean_acc_rms'] for r in recordings]
groups   = [r['group']        for r in recordings]
colors   = [GROUP_COLORS[g]   for g in groups]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(rec_ids, mean_rms, color=colors, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Recording ID')
ax.set_ylabel('Mean RMS ACC magnitude (g)')
ax.set_title('Per-recording mean ACC magnitude — T1 treadmill / T2 mixed-arm / T3 boxing')
ax.set_xticks(rec_ids)
ax.set_xticklabels([str(i) for i in rec_ids], fontsize=8)

handles = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in ('T1', 'T2', 'T3')]
ax.legend(handles=handles, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

for grp in ('T1', 'T2', 'T3'):
    vals = [r['mean_acc_rms'] for r in recordings if r['group'] == grp]
    print(f'{GROUP_LABELS[grp]:30s}  n={len(vals):2d}  mean ACC RMS: {np.mean(vals):.3f}  (range {min(vals):.3f}–{max(vals):.3f})')


## 1.4 — Sanity check: raw signal plots

Three 4-panel figures — one representative recording from each group — first 30 s.

**Expected output:**
- Panel 1 (ECG): T1 shows classic QRS complexes at ~1–2 Hz. T2 and T3 show an "ECG not available (test set)" placeholder.
- Panel 2 (PPG): two channels overlaid — roughly periodic AC-coupled pulse waveform, often with motion-induced distortion visible on top (especially in T3 boxing).
- Panel 3 (ACC x/y/z):
  - T1: clean periodic cycles at running cadence (1.5–3 Hz)
  - T2: irregular bursts separated by quieter segments (rehab-style movement)
  - T3: sustained high-frequency irregular activity (boxing punches)
- Panel 4 (ACC magnitude):
  - T1: rhythmic oscillation above ~1 g
  - T2: punctuated spikes
  - T3: continuously noisy, highest amplitude

If any panel is flat for a group where it shouldn't be, or the PPG looks like pure noise with no visible pulses even at rest, flag it.


In [ ]:
def plot_raw(rec, duration_s=30):
    """Plot ECG (if present), PPG (both channels), ACC axes, and ACC magnitude."""
    n   = int(duration_s * FS)
    t   = np.arange(n) / FS
    ecg = rec['ecg']
    ppg = rec['ppg']
    acc = rec['acc']

    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(f"Recording {rec['rec_id']:02d} ({GROUP_LABELS[rec['group']]}) "
                 f"— first {duration_s} s",
                 fontsize=11)

    if ecg is not None:
        axes[0].plot(t, ecg[:n], lw=0.6, color='k')
    else:
        axes[0].text(0.5, 0.5, 'ECG not available (test set)',
                     ha='center', va='center', transform=axes[0].transAxes,
                     fontsize=10, color='grey')
    axes[0].set_ylabel('ECG')

    axes[1].plot(t, ppg[0, :n], lw=0.6, color='royalblue',       label='PPG ch1')
    axes[1].plot(t, ppg[1, :n], lw=0.6, color='cornflowerblue', alpha=0.7, label='PPG ch2')
    axes[1].set_ylabel('PPG')
    axes[1].legend(fontsize=7, loc='upper right')

    for i, (label, color) in enumerate(zip(['ACC x', 'ACC y', 'ACC z'],
                                           ['crimson', 'darkorange', 'forestgreen'])):
        axes[2].plot(t, acc[i, :n], lw=0.6, label=label, color=color)
    axes[2].set_ylabel('ACC (3 axes)')
    axes[2].legend(fontsize=7, loc='upper right')

    mag = np.sqrt(np.sum(acc[:, :n]**2, axis=0))
    axes[3].plot(t, mag, lw=0.6, color=GROUP_COLORS[rec['group']])
    axes[3].set_ylabel('ACC magnitude')
    axes[3].set_xlabel('Time (s)')

    plt.tight_layout()
    plt.show()


# Eyeball one representative recording from each group
for grp in ('T1', 'T2', 'T3'):
    example = next(r for r in recordings if r['group'] == grp)
    plot_raw(example)


## 1.5 — Window count vs GT count check

The HR estimator in Step 2 needs 1:1 alignment between our 8 s/2 s-shift windows and the `BPM0` ground-truth values. We verify that here.

**Expected output:**
- Table of 22 rows: `Rec | Group | Windows | GT vals | Status`
- Every row should say `OK` (`|n_windows - n_gt| ≤ 1`)
- Final line: `Mismatched recordings: 0 / 22`
- Any `MISMATCH` row is a red flag — stop and investigate before moving to Step 2


In [ ]:
# Note: the outer string uses double quotes so the inner 'Rec' / 'Group' single-quoted
# keys parse cleanly under Python 3.11 (which disallows same-quote nesting in f-strings).
header = f"{'Rec':>4}  {'Group':>6}  {'Windows':>8}  {'GT vals':>8}  Status"
print(header)
print('-' * len(header))

n_mismatch = 0
for r in recordings:
    diff   = abs(r['n_windows'] - r['n_gt'])
    status = 'OK' if diff <= 1 else 'MISMATCH'
    flag   = '' if diff <= 1 else '  ← MISMATCH'
    if diff > 1:
        n_mismatch += 1
    print(f"{r['rec_id']:>4}  {r['group']:>6}  {r['n_windows']:>8}  {r['n_gt']:>8}  {status}{flag}")

print()
print(f"Mismatched recordings: {n_mismatch} / {len(recordings)}")
